In [2]:
# ============================================
# DAY 5: FINE-TUNED LLM-POWERED CHATBOT
# CELL 1: SETUP, IMPORTS & ENVIRONMENT
# Tech Prime Pvt Limited - Advanced AI/ML Internship
# ============================================

print("="*70)
print("DAY 5: FINE-TUNED LLM-POWERED CHATBOT")
print("CELL 1: SETUP & ENVIRONMENT")
print("="*70)

# ============================================
# STEP 1: IMPORTS
# ============================================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import pickle
import random
from datetime import datetime
from typing import List, Dict, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Hugging Face Libraries
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    pipeline,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset, Dataset, concatenate_datasets

# PEFT for LoRA
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    TaskType,
    prepare_model_for_kbit_training
)

# Gradio for UI
import gradio as gr

print(f"\n✓ All imports loaded successfully!")

# ============================================
# STEP 2: CHECK ENVIRONMENT
# ============================================
print(f"\n📊 Environment Info:")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ============================================
# STEP 3: CREATE DIRECTORIES
# ============================================
os.makedirs("models", exist_ok=True)
os.makedirs("static", exist_ok=True)
os.makedirs("templates", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("chatbot_model", exist_ok=True)

print("\n✓ Project directories created!")

# ============================================
# STEP 4: LOAD TOKENIZER (DialoGPT)
# ============================================
print("\n📚 Loading DialoGPT Tokenizer...")

# Use DialoGPT-medium for better conversations
model_name = "microsoft/DialoGPT-medium"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✓ Tokenizer loaded successfully!")
    print(f"  Model: {model_name}")
    print(f"  Vocabulary size: {tokenizer.vocab_size:,}")
except Exception as e:
    print(f"⚠️ Could not load DialoGPT tokenizer: {e}")
    print("Falling back to GPT-2...")
    model_name = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✓ Using GPT-2 tokenizer")
    print(f"  Vocabulary size: {tokenizer.vocab_size:,}")

# ============================================
# STEP 5: LOAD CONVERSATIONAL DATASETS
# ============================================
print("\n" + "="*50)
print("📊 LOADING CONVERSATIONAL DATASETS")
print("="*50)

# Dataset 1: BlendedSkillTalk
print("\n1. Loading BlendedSkillTalk...")
try:
    bst = load_dataset("blended_skill_talk")
    print(f"   ✓ BlendedSkillTalk loaded!")
    print(f"     - Train: {len(bst['train'])} samples")
    print(f"     - Validation: {len(bst['validation'])} samples")
    print(f"     - Test: {len(bst['test'])} samples")
except Exception as e:
    print(f"   ✗ Failed: {e}")
    bst = None

# Dataset 2: EmpatheticDialogues
print("\n2. Loading EmpatheticDialogues...")
try:
    empathy = load_dataset("empathetic_dialogues")
    print(f"   ✓ EmpatheticDialogues loaded!")
    print(f"     - Train: {len(empathy['train'])} samples")
    print(f"     - Validation: {len(empathy['validation'])} samples")
    print(f"     - Test: {len(empathy['test'])} samples")
except Exception as e:
    print(f"   ✗ Failed: {e}")
    empathy = None

# Dataset 3: DailyDialog (if available)
print("\n3. Loading DailyDialog...")
try:
    daily = load_dataset("daily_dialog", trust_remote_code=True)
    print(f"   ✓ DailyDialog loaded!")
    print(f"     - Train: {len(daily['train'])} samples")
    print(f"     - Validation: {len(daily['validation'])} samples")
    print(f"     - Test: {len(daily['test'])} samples")
except Exception as e:
    print(f"   ✗ Failed: {e}")
    daily = None

# ============================================
# STEP 6: PREVIEW DATA
# ============================================
print("\n" + "="*50)
print("📝 DATA PREVIEW")
print("="*50)

if bst:
    print("\nBlendedSkillTalk Sample:")
    sample = bst['train'][0]
    if 'free_messages' in sample and sample['free_messages']:
        print(f"  Conversation: {sample['free_messages'][:3]}")

if empathy:
    print("\nEmpatheticDialogues Sample:")
    sample = empathy['train'][0]
    if 'utterance' in sample:
        print(f"  Utterance: {sample['utterance'][:100]}...")

# ============================================
# STEP 7: CREATE DATA EXTRACTION FUNCTIONS
# ============================================
print("\n📝 Creating data extraction functions...")

def extract_blended_skill_conversations(data, max_samples=200):
    """Extract conversations from BlendedSkillTalk"""
    conversations = []
    for item in data[:max_samples]:
        messages = []
        if 'free_messages' in item and item['free_messages']:
            for msg in item['free_messages']:
                if msg and msg.strip():
                    messages.append(msg.strip())
        
        if messages and len(messages) >= 2:
            conv = []
            for i, msg in enumerate(messages[:8]):
                if i % 2 == 0:
                    conv.append(f"Human: {msg}")
                else:
                    conv.append(f"Assistant: {msg}")
            conversations.append(" ".join(conv))
    return conversations

def extract_empathetic_conversations(data, max_samples=200):
    """Extract conversations from EmpatheticDialogues"""
    conversations = []
    for i in range(min(max_samples, len(data))):
        item = data[i]
        if 'utterance' in item:
            msg = item['utterance']
            if msg and msg.strip():
                conv = []
                if 'context' in item and item['context']:
                    context = item['context']
                    if context:
                        conv.append(f"Human: {context}")
                conv.append(f"Assistant: {msg}")
                if len(conv) >= 2:
                    conversations.append(" ".join(conv))
    return conversations

print("✓ Data extraction functions ready!")

# ============================================
# STEP 8: SUMMARY
# ============================================
print("\n" + "="*50)
print("📋 CELL 1 SUMMARY")
print("="*50)

print(f"""
✅ Environment Setup Complete!
✅ Tokenizer Loaded: {model_name}
✅ Datasets Available:
   - BlendedSkillTalk: {'✓' if bst else '✗'}
   - EmpatheticDialogues: {'✓' if empathy else '✗'}
   - DailyDialog: {'✓' if daily else '✗'}

✅ Directories Created:
   - models/
   - static/
   - templates/
   - logs/
   - data/
   - chatbot_model/

Next Steps:
- Cell 2: Load DialoGPT Model with LoRA
- Cell 3: Prepare Dataset with Personal Data
- Cell 4: Train the Model (ACTUAL TRAINING!)
- Cell 5: Test and Deploy

⚠️ IMPORTANT: We will ACTUALLY TRAIN in Cell 4!
""")

print("="*70)
print("CELL 1 COMPLETE - Ready for Next Step!")
print("="*70)

DAY 5: FINE-TUNED LLM-POWERED CHATBOT
CELL 1: SETUP & ENVIRONMENT



✓ All imports loaded successfully!

📊 Environment Info:
  PyTorch version: 2.10.0+cu128
  CUDA available: True
  GPU: Tesla T4
  GPU Memory: 15.64 GB

✓ Project directories created!

📚 Loading DialoGPT Tokenizer...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'daily_dialog' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


   ✗ Failed: Dataset scripts are no longer supported, but found empathetic_dialogues.py

3. Loading DailyDialog...


README.md: 0.00B [00:00, ?B/s]

daily_dialog.py: 0.00B [00:00, ?B/s]

   ✗ Failed: Dataset scripts are no longer supported, but found daily_dialog.py

📝 DATA PREVIEW

BlendedSkillTalk Sample:
  Conversation: ['I like acting, I hope to be an actor, what about you?', 'No, but someday.', 'After I am done with school I plan to have a family.']

📝 Creating data extraction functions...
✓ Data extraction functions ready!

📋 CELL 1 SUMMARY

✅ Environment Setup Complete!
✅ Tokenizer Loaded: microsoft/DialoGPT-medium
✅ Datasets Available:
   - BlendedSkillTalk: ✓
   - EmpatheticDialogues: ✗
   - DailyDialog: ✗

✅ Directories Created:
   - models/
   - static/
   - templates/
   - logs/
   - data/
   - chatbot_model/

Next Steps:
- Cell 2: Load DialoGPT Model with LoRA
- Cell 3: Prepare Dataset with Personal Data
- Cell 4: Train the Model (ACTUAL TRAINING!)
- Cell 5: Test and Deploy

⚠️ IMPORTANT: We will ACTUALLY TRAIN in Cell 4!

CELL 1 COMPLETE - Ready for Next Step!


In [3]:
# CELL 2: LOAD MODEL WITH LORA (FIXED - USING SMALLER MODEL)

print("="*70)
print("CELL 2: LOAD MODEL WITH LORA (FIXED)")
print("="*70)

import os
os.environ["PEFT_USE_TORCHAO"] = "0"

import peft.tuners.lora.torchao
original_is_available = peft.tuners.lora.torchao.is_torchao_available
peft.tuners.lora.torchao.is_torchao_available = lambda: False

print("\nLoading DialoGPT-small model (124M parameters)...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")
print(f"DialoGPT-small loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

model.to(device)
print(f"Model on: {device}")

# Clear memory
torch.cuda.empty_cache()
print(f"GPU Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\nConfiguring LoRA...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")

print("\nApplying LoRA to model...")

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

total_params = sum(p.numel() for p in peft_model.parameters())
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

print(f"\nParameter Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable: {trainable_params/total_params*100:.3f}%")
print(f"  Parameter reduction: {(1 - trainable_params/total_params)*100:.2f}%")

peft.tuners.lora.torchao.is_torchao_available = original_is_available

print("\n" + "="*70)
print("CELL 2 COMPLETE - Model Ready!")
print("="*70)

CELL 2: LOAD MODEL WITH LORA (FIXED)

Loading DialoGPT-small model (124M parameters)...


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

DialoGPT-small loaded successfully!
Total parameters: 163,037,184
Model on: cuda
GPU Memory used: 0.35 GB

Configuring LoRA...
  Rank (r): 8
  Alpha: 16
  Target modules: {'c_proj', 'c_fc', 'c_attn'}
  Dropout: 0.1

Applying LoRA to model...
trainable params: 1,179,648 || all params: 164,216,832 || trainable%: 0.7183

Parameter Statistics:
  Total parameters: 164,216,832
  Trainable parameters: 1,179,648
  Trainable: 0.718%
  Parameter reduction: 99.28%

CELL 2 COMPLETE - Model Ready!


In [4]:
# CELL 3: PREPARE DATASET WITH PERSONAL DATA

print("="*70)
print("CELL 3: PREPARE DATASET WITH PERSONAL DATA")
print("="*70)

print("\nExtracting conversations from BlendedSkillTalk...")

def extract_blended_skill(data, max_samples=500):
    conversations = []
    for idx in range(min(max_samples, len(data))):
        item = data[idx]
        messages = []
        if 'free_messages' in item and item['free_messages']:
            for msg in item['free_messages']:
                if msg and msg.strip():
                    messages.append(msg.strip())
        
        if messages and len(messages) >= 2:
            conv = []
            for i, msg in enumerate(messages[:8]):
                if i % 2 == 0:
                    conv.append(f"Human: {msg}")
                else:
                    conv.append(f"Assistant: {msg}")
            if conv:
                conversations.append({"text": " ".join(conv)})
    return conversations

blended_conversations = extract_blended_skill(bst['train'], max_samples=500)
print(f"  Extracted {len(blended_conversations)} conversations from BlendedSkillTalk")

print("\nCreating personalized conversations...")

PERSONAL_INFO = {
    "name": "Sanaullah",
    "education": "BSCS student at Abasyn University Islamabad",
    "cgpa": "3.86/4.00",
    "skills": "Python, Pandas, NumPy, Matplotlib, PyTorch, CV, CNN",
    "projects": "AI Chatbot",
    "internship": "AI/ML Engineer Intern at Tech Prime Pvt. Limited",
    "interests": "Data Science, Machine Learning, Artificial Intelligence",
    "certifications": "Kaggle Python, Udemy, IBM, HP Foundation, Mind Labs AI",
    "github": "https://github.com/SANAULLAH-AI",
    "linkedin": "https://www.linkedin.com/in/sanaullah-ai",
    "portfolio": "https://sanaullah7964.netlify.app/"
}

personalized_data = []

qa_pairs = [
    ("Who created you?", f"My creator is {PERSONAL_INFO['name']}."),
    ("Who made you?", f"{PERSONAL_INFO['name']} created me."),
    ("Tell me about your creator.", f"{PERSONAL_INFO['name']} is a {PERSONAL_INFO['education']} with CGPA {PERSONAL_INFO['cgpa']}."),
    ("What is your creator's name?", f"{PERSONAL_INFO['name']}."),
    ("What does your creator do?", f"{PERSONAL_INFO['name']} is {PERSONAL_INFO['internship']}."),
    ("What skills does your creator have?", f"{PERSONAL_INFO['name']} knows {PERSONAL_INFO['skills']}."),
    ("What projects did your creator build?", f"{PERSONAL_INFO['name']} built {PERSONAL_INFO['projects']}."),
    ("What certifications does your creator have?", f"{PERSONAL_INFO['name']} has {PERSONAL_INFO['certifications']}."),
    ("What is your creator's CGPA?", f"{PERSONAL_INFO['cgpa']}."),
    ("Where does your creator study?", f"{PERSONAL_INFO['name']} studies at {PERSONAL_INFO['education']}."),
    ("What is your creator's GitHub?", f"{PERSONAL_INFO['github']}"),
    ("What is your creator's LinkedIn?", f"{PERSONAL_INFO['linkedin']}"),
    ("What is your creator's portfolio?", f"{PERSONAL_INFO['portfolio']}"),
    ("What are your creator's interests?", f"{PERSONAL_INFO['name']} is interested in {PERSONAL_INFO['interests']}."),
]

for q, a in qa_pairs:
    personalized_data.append({"text": f"Human: {q} Assistant: {a}"})
    personalized_data.append({"text": f"Human: Can you tell me {q.lower()}? Assistant: {a}"})
    personalized_data.append({"text": f"Human: I'd like to know {q.lower()}. Assistant: {a}"})
    personalized_data.append({"text": f"Human: Please tell me {q.lower()}. Assistant: {a}"})

general = [
    ("Hello!", f"Hello! I'm {PERSONAL_INFO['name']}'s AI assistant. How can I help?"),
    ("Hi there!", f"Hi! Great to chat with you. I'm {PERSONAL_INFO['name']}'s chatbot."),
    ("How are you?", f"I'm doing great! Thanks for asking."),
    ("What's your name?", f"I'm {PERSONAL_INFO['name']}'s AI Assistant."),
    ("What can you do?", f"I can chat with you and answer questions about {PERSONAL_INFO['name']}."),
    ("Tell me a joke.", "Why don't scientists trust atoms? Because they make up everything!"),
    ("Goodbye!", "Goodbye! Have a wonderful day!"),
]

for q, a in general:
    personalized_data.append({"text": f"Human: {q} Assistant: {a}"})
    personalized_data.append({"text": f"Human: {q}? Assistant: {a}"})

print(f"  Created {len(personalized_data)} personalized conversations")

print("\nCombining all data...")

all_data = blended_conversations + personalized_data
print(f"  Total training samples: {len(all_data)}")
print(f"  - BlendedSkillTalk: {len(blended_conversations)}")
print(f"  - Personalized: {len(personalized_data)}")

print("\nSample training data:")
print("-"*50)
for i in range(min(2, len(all_data))):
    text = all_data[i]['text'][:120] + "..." if len(all_data[i]['text']) > 120 else all_data[i]['text']
    print(f"{i+1}. {text}")
print("-"*50)

print("\nCreating Dataset...")

from datasets import Dataset

train_dataset = Dataset.from_list(all_data)
train_dataset = train_dataset.shuffle(seed=42)

train_size = int(0.9 * len(train_dataset))
train_split = Dataset.from_dict({'text': train_dataset['text'][:train_size]})
val_split = Dataset.from_dict({'text': train_dataset['text'][train_size:]})

print(f"  Training samples: {len(train_split)}")
print(f"  Validation samples: {len(val_split)}")

print("\nTokenizing datasets...")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=128,
        padding=False
    )

train_split = train_split.map(tokenize_function, batched=True, remove_columns=['text'])
val_split = val_split.map(tokenize_function, batched=True, remove_columns=['text'])

train_split.set_format('torch', columns=['input_ids', 'attention_mask'])
val_split.set_format('torch', columns=['input_ids', 'attention_mask'])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("  Tokenization complete!")

print("\n" + "="*70)
print("CELL 3 COMPLETE - Dataset Ready for Training!")
print("="*70)

CELL 3: PREPARE DATASET WITH PERSONAL DATA

Extracting conversations from BlendedSkillTalk...
  Extracted 482 conversations from BlendedSkillTalk

Creating personalized conversations...
  Created 70 personalized conversations

Combining all data...
  Total training samples: 552
  - BlendedSkillTalk: 482
  - Personalized: 70

Sample training data:
--------------------------------------------------
1. Human: I like acting, I hope to be an actor, what about you? Assistant: No, but someday. Human: After I am done with sch...
2. Human: Oh really!? That is interesting. I am actually italian myself. Assistant: Moving in a new place can be a lot of f...
--------------------------------------------------

Creating Dataset...
  Training samples: 496
  Validation samples: 56

Tokenizing datasets...


Map:   0%|          | 0/496 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

  Tokenization complete!

CELL 3 COMPLETE - Dataset Ready for Training!


In [5]:
# CELL 4: TRAIN THE MODEL (ACTUAL TRAINING)

print("="*70)
print("CELL 4: TRAIN THE MODEL (ACTUAL TRAINING)")
print("="*70)

print("\nStarting actual training with LoRA...")

training_args = TrainingArguments(
    output_dir="./chatbot_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=2e-4,
    report_to="none",
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
)

print(f"Training Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch Size: {training_args.per_device_train_batch_size}")
print(f"  Effective Batch Size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning Rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")
print(f"  Training Samples: {len(train_split)}")
print(f"  Validation Samples: {len(val_split)}")

print("\nCreating trainer...")

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_split,
    data_collator=data_collator,
)

print("Trainer created successfully!")

print("\n" + "="*50)
print("STARTING TRAINING...")
print("="*50)
print(f"Training on {len(train_split)} samples for {training_args.num_train_epochs} epochs")
print("This will take approximately 15-20 minutes...")

# UNCOMMENT THE LINE BELOW TO ACTUALLY TRAIN
trainer.train()

print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)

print("\nSaving model...")

peft_model.save_pretrained("./chatbot_model/lora_adapter")
tokenizer.save_pretrained("./chatbot_model/tokenizer")

model_info = {
    "model": "DialoGPT-medium",
    "lora_rank": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "trainable_params": trainable_params,
    "total_params": total_params,
    "train_samples": len(train_split),
    "epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "creator": PERSONAL_INFO["name"]
}

with open("./chatbot_model/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("Model saved to: ./chatbot_model")
print("  - LoRA adapter: ./chatbot_model/lora_adapter")
print("  - Tokenizer: ./chatbot_model/tokenizer")
print("  - Model info: ./chatbot_model/model_info.json")

print("\n" + "="*70)
print("CELL 4 COMPLETE - Model Trained and Saved!")
print("="*70)

CELL 4: TRAIN THE MODEL (ACTUAL TRAINING)

Starting actual training with LoRA...
Training Configuration:
  Epochs: 3
  Batch Size: 2
  Effective Batch Size: 8
  Learning Rate: 0.0002
  FP16: True
  Training Samples: 496
  Validation Samples: 56

Creating trainer...
Trainer created successfully!

STARTING TRAINING...
Training on 496 samples for 3 epochs
This will take approximately 15-20 minutes...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
50,4.609848,4.112903



TRAINING COMPLETE!

Saving model...
Model saved to: ./chatbot_model
  - LoRA adapter: ./chatbot_model/lora_adapter
  - Tokenizer: ./chatbot_model/tokenizer
  - Model info: ./chatbot_model/model_info.json

CELL 4 COMPLETE - Model Trained and Saved!


In [6]:
# CELL 5: TEST AND DEPLOY CHATBOT (FIXED)

print("="*70)
print("CELL 5: TEST AND DEPLOY CHATBOT (FIXED)")
print("="*70)

import os
os.environ["PEFT_USE_TORCHAO"] = "0"

import peft.tuners.lora.torchao
original_is_available = peft.tuners.lora.torchao.is_torchao_available
peft.tuners.lora.torchao.is_torchao_available = lambda: False

print("\nLoading trained model...")

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

try:
    model_trained = PeftModel.from_pretrained(base_model, "./chatbot_model/lora_adapter")
    print("LoRA adapter loaded successfully!")
except Exception as e:
    print(f"Could not load adapter: {e}")
    print("Using base model...")
    model_trained = base_model

model_trained.to(device)
model_trained.eval()

peft.tuners.lora.torchao.is_torchao_available = original_is_available

print("\nTesting model responses...")

def generate_response(prompt, max_len=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_trained.generate(
            inputs.input_ids,
            max_length=len(inputs.input_ids[0]) + max_len,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
            top_p=0.9
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Assistant:" in response:
        response = response.split("Assistant:")[-1].strip()
    response = response.split("Human:")[0].strip()
    return response

test_questions = [
    "Human: Who created you? Assistant:",
    "Human: Hello! How are you? Assistant:"
]

print("\nSample Responses:")
for q in test_questions:
    response = generate_response(q)
    print(f"\nPrompt: {q}")
    print(f"Response: {response[:150]}")

print("\nCreating Gradio UI...")

def chat_fn(message, history):
    if not message or message.strip() == "":
        return history
    prompt = f"Human: {message} Assistant:"
    response = generate_response(prompt)
    history.append((message, response))
    return history

def reset_fn():
    return []

with gr.Blocks(title="Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AI Chatbot Assistant")
    gr.Markdown("Powered by DialoGPT + LoRA | Created by Sanaullah")
    
    chatbot_ui = gr.Chatbot(height=400, bubble_full_width=False, avatar_images=("User", "AI"))
    
    with gr.Row():
        msg = gr.Textbox(placeholder="Type your message...", scale=4, lines=1)
        send_btn = gr.Button("Send", variant="primary", scale=1)
    
    reset_btn = gr.Button("Reset")
    
    with gr.Accordion("Example Questions", open=False):
        gr.Examples(
            examples=[
                ["Who created you?"],
                ["Tell me about your creator."],
                ["What is your creator's name?"],
                ["What skills does your creator have?"],
                ["Hello! How are you?"]
            ],
            inputs=msg
        )
    
    msg.submit(chat_fn, [msg, chatbot_ui], [chatbot_ui]).then(lambda: "", None, [msg])
    send_btn.click(chat_fn, [msg, chatbot_ui], [chatbot_ui]).then(lambda: "", None, [msg])
    reset_btn.click(reset_fn, None, [chatbot_ui])

print("UI created successfully!")

print("\nLaunching chatbot...")

# Kill existing processes
import subprocess
try:
    subprocess.run(['fuser', '-k', '7860/tcp'], capture_output=True)
    print("Port 7860 freed")
except:
    pass

try:
    from pyngrok import ngrok
    ngrok.set_auth_token("2FxKFBPqJZnBeUC5s3krTDqwkge_6yt7XizoLyhqqDmHUZq8U")
    ngrok.kill()
    public_url = ngrok.connect(7860)
    print(f"Public URL: {public_url}")
except Exception as e:
    print(f"Ngrok error: {e}")

try:
    demo.queue()
    demo.launch(share=True, server_name="0.0.0.0", server_port=7860)
except OSError:
    print("Port 7860 busy, trying 7861...")
    demo.launch(share=True, server_name="0.0.0.0", server_port=7861)

print("\n" + "="*70)
print("CELL 5 COMPLETE - Chatbot Deployed!")
print("="*70)

CELL 5: TEST AND DEPLOY CHATBOT (FIXED)

Loading trained model...


pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Could not load adapter: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight: copying a param with shape torch.Size([8, 768]) from checkpoint, the shape in current model is torch.Size([8, 1024]).
	size mismatch for base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight: copying a param with shape torch.Size([2304, 8]) from checkpoint, the shape in current model is torch.Size([3072, 8]).
	size mismatch for base_model.model.transformer.h.0.attn.c_proj.lora_A.default.weight: copying a param with shape torch.Size([8, 768]) from checkpoint, the shape in current model is torch.Size([8, 1024]).
	size mismatch for base_model.model.transformer.h.0.attn.c_proj.lora_B.default.weight: copying a param with shape torch.Size([768, 8]) from checkpoint, the shape in current model is torch.Size([1024, 8]).
	size mismatch for base_model.model.transformer.h.0.mlp.c_fc.lora_A.default.weight: copying a para

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Testing model responses...

Sample Responses:

Prompt: Human: Who created you? Assistant:
Response: It was me! Human : No, I didn't.

Prompt: Human: Hello! How are you? Assistant:
Response: Oh, I'm fine.

Creating Gradio UI...
UI created successfully!

Launching chatbot...
Port 7860 freed
Ngrok error: No module named 'pyngrok'
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://5df2ac4c58676a90e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



CELL 5 COMPLETE - Chatbot Deployed!
